# 001 · Double-Ended Queues with deque

> **Standard Library › Data Structures** · Python 3.14

`collections.deque` ("deck") is a double-ended queue: O(1) appends and pops at **both** ends. It's Python's answer to `Queue<T>`, `Stack<T>` and most uses of `LinkedList<T>` — plus a built-in bounded ring buffer.

**In this notebook you will learn:**

- how `deque` maps to `Queue<T>`, `Stack<T>` and `LinkedList<T>`
- the O(1) operations: `append`, `appendleft`, `pop`, `popleft`
- using `maxlen` as a ring buffer ("keep the last N items")
- `rotate`, `extendleft` and the other extras
- why `list.pop(0)` is a performance trap, and when to use `queue.Queue` instead

## .NET mental model

| .NET | Python |
|---|---|
| `Queue<T>` — `Enqueue` / `Dequeue` / `Peek` | `deque` — `append` / `popleft` / `d[0]` |
| `Stack<T>` — `Push` / `Pop` / `Peek` | `list` (or `deque`) — `append` / `pop` / `s[-1]` |
| `LinkedList<T>` — `AddFirst` / `AddLast` / `RemoveFirst` / `RemoveLast` | `deque` — `appendleft` / `append` / `popleft` / `pop` |
| fixed-size circular buffer (hand-written) | `deque(maxlen=n)` |
| `ConcurrentQueue<T>` / `BlockingCollection<T>` / `Channel<T>` | `queue.Queue` / `asyncio.Queue` |

One type covers all three .NET collections. Internally it is a doubly-linked list of fixed-size blocks, so both ends are cheap but the middle is not.

## Creating a deque

```csharp
// C#
var queue = new Queue<string>(new[] { "a", "b", "c" });
```

In [ ]:
from collections import deque

empty = deque()
letters = deque(["a", "b", "c"])     # from any iterable
chars = deque("xyz")                 # a string is an iterable of characters

print(empty, letters, chars)
print(len(letters), "b" in letters)  # len() and `in` work as usual

## As a FIFO queue

`append` adds to the right; `popleft` removes from the left. Peek with `d[0]`.

```csharp
// C#
var q = new Queue<string>();
q.Enqueue("job-1"); q.Enqueue("job-2");
var next = q.Dequeue();   // "job-1"
var peek = q.Peek();      // "job-2"
```

In [ ]:
jobs = deque()
jobs.append("job-1")            # Enqueue
jobs.append("job-2")
jobs.append("job-3")

print(jobs.popleft())           # Dequeue -> job-1
print(jobs[0])                  # Peek    -> job-2
print(jobs)

A common pattern is a processing loop. An empty deque is falsy, so `while queue:` reads naturally — this is the backbone of breadth-first search:

In [ ]:
graph = {"A": ["B", "C"], "B": ["D"], "C": ["D", "E"], "D": [], "E": []}

def bfs(start: str) -> list[str]:
    seen, order, queue = {start}, [], deque([start])
    while queue:
        node = queue.popleft()
        order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return order

print(bfs("A"))

## As a LIFO stack

`append` + `pop` work from the right end. For a pure stack a plain `list` is just as good (its `append`/`pop` are also O(1)) and is what most Python code uses; reach for `deque` when you need both ends.

```csharp
// C#
var stack = new Stack<int>();
stack.Push(1); stack.Push(2);
var top = stack.Pop();   // 2
```

In [ ]:
stack = deque()
stack.append(1)    # Push
stack.append(2)
print(stack.pop()) # Pop -> 2
print(stack[-1])   # Peek -> 1

## Both ends: `appendleft` and `popleft`

All four end-operations are O(1) — like `LinkedList<T>.AddFirst/AddLast/RemoveFirst/RemoveLast`, but without node objects.

In [ ]:
d = deque([2, 3])
d.appendleft(1)        # AddFirst
d.append(4)            # AddLast
print(d)
print(d.popleft(), d.pop())   # RemoveFirst, RemoveLast
print(d)

⚠️ **Gotcha:** `extendleft` adds items **one at a time** to the left, so they end up in **reverse** order.

In [ ]:
d = deque([1, 2, 3])
d.extend([4, 5])        # right side, order kept
d.extendleft([0, -1])   # left side: -1 ends up first!
print(d)

⚠️ **Gotcha:** popping from an empty deque raises `IndexError` (C#'s `Dequeue` throws `InvalidOperationException`). There is no `TryDequeue` — check truthiness first.

In [ ]:
d = deque()
try:
    d.popleft()
except IndexError as e:
    print(f"{type(e).__name__}: {e}")

item = d.popleft() if d else None   # "TryDequeue"
print(item)

## `maxlen`: a ring buffer for free

With `maxlen=n` the deque never grows beyond `n` items: appending to a full deque silently **drops an item from the opposite end**. Perfect for "the last N log lines", moving averages, or recent-history features. .NET has no built-in equivalent.

In [ ]:
recent = deque(maxlen=3)
for reading in [10, 20, 30, 40, 50]:
    recent.append(reading)
    print(list(recent))

print(recent.maxlen)

In [ ]:
# Moving average over the last 3 sensor readings
window = deque(maxlen=3)
for value in [4, 8, 6, 10, 2]:
    window.append(value)
    print(value, "->", round(sum(window) / len(window), 2))

**Tip:** "keep the last N lines of a file" is a one-liner inside `with open(path, encoding="utf-8") as f:` — `deque(f, maxlen=N)`. The deque consumes the iterable and only remembers the tail (like `tail -n N`).

## `rotate`

`rotate(n)` shifts items `n` steps to the right (negative = left), wrapping around. Useful for round-robin scheduling.

In [ ]:
d = deque("abcde")
d.rotate(2)
print(d)
d.rotate(-3)
print(d)

In [ ]:
# Round-robin: serve the front worker, then move it to the back
workers = deque(["w1", "w2", "w3"])
for task in range(5):
    print(f"task {task} -> {workers[0]}")
    workers.rotate(-1)

## Other operations

`deque` also supports `insert`, `remove`, `index`, `count`, `reverse`, `clear`, `copy` and indexing with `d[i]` — but these are **O(n)** (indexing is O(1) at the ends, slower towards the middle). Slicing is **not** supported.

In [ ]:
d = deque([1, 2, 3, 2])
print(d.count(2), d.index(3))
d.remove(2)          # first occurrence
d.reverse()
print(d)

try:
    d[0:2]
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

import itertools
print(list(itertools.islice(d, 0, 2)))   # the workaround

## Gotcha: `list.pop(0)` is O(n)

A `list` is a dynamic array — like `List<T>`. Removing the first element shifts every other element left, exactly like `List<T>.RemoveAt(0)`. Using a list as a FIFO queue is therefore **quadratic**.

In [ ]:
import timeit

n = 50_000

def drain_list():
    items = list(range(n))
    while items:
        items.pop(0)          # O(n) each time

def drain_deque():
    items = deque(range(n))
    while items:
        items.popleft()       # O(1) each time

t_list = timeit.timeit(drain_list, number=1)
t_deque = timeit.timeit(drain_deque, number=1)
print(f"list.pop(0):     {t_list:.4f}s")
print(f"deque.popleft(): {t_deque:.4f}s")
print("deque is faster:", t_deque < t_list)   # exact numbers vary by machine

The same applies to `list.insert(0, x)`. If you find yourself writing either, switch to `deque`.

## Not a thread-safe work queue

In CPython, `append` and `popleft` on a deque are atomic, so a deque won't get corrupted by concurrent use. But it has **no blocking operations**: a consumer can't *wait* for an item, and there's no `task_done`/`join`. For producer/consumer pipelines use `queue.Queue` (the `BlockingCollection<T>` analogue) or `asyncio.Queue` (≈ `Channel<T>`) *(see Standard Library › Concurrency › 002 · Locks, Events and Queues)*.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Create | `new Queue<T>(items)` | `deque(items)` |
| Enqueue | `q.Enqueue(x)` | `d.append(x)` |
| Dequeue | `q.Dequeue()` | `d.popleft()` |
| Peek front | `q.Peek()` | `d[0]` |
| Push (stack) | `s.Push(x)` | `d.append(x)` (or `list.append`) |
| Pop (stack) | `s.Pop()` | `d.pop()` |
| Add at front | `ll.AddFirst(x)` | `d.appendleft(x)` |
| Bounded ring buffer | hand-written | `deque(maxlen=n)` |
| Rotate | hand-written | `d.rotate(n)` |
| Empty check | `q.Count == 0` | `not d` |
| Blocking queue | `BlockingCollection<T>` / `Channel<T>` | `queue.Queue` / `asyncio.Queue` |

## Exercises

1. Write `tail(lines, n)` that returns the last `n` items of any iterable without building a full list.
2. Check whether a string is a palindrome (ignoring case and non-letters) by popping from both ends of a deque.
3. Simulate a printer queue: jobs `["doc1", "doc2", "doc3"]` arrive; each "tick" the printer prints one page of the front job. Each job has 2 pages. Rotate unfinished jobs to the back (round-robin) and print the order in which pages are printed.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from collections import deque

# 1 — maxlen keeps only the tail
def tail(lines, n):
    return list(deque(lines, maxlen=n))

print(tail(range(100), 3))

# 2
def is_palindrome(s: str) -> bool:
    d = deque(ch.casefold() for ch in s if ch.isalpha())
    while len(d) > 1:
        if d.popleft() != d.pop():
            return False
    return True

print(is_palindrome("Never odd or even"), is_palindrome("deque"))

# 3 — (name, pages_left) tuples, round-robin
queue = deque((name, 2) for name in ["doc1", "doc2", "doc3"])
while queue:
    name, pages = queue.popleft()
    print(f"printing {name} page {3 - pages}")
    if pages > 1:
        queue.append((name, pages - 1))
```

</details>